In [10]:
%pip install google-adk litellm ollama

import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from google.adk.tools.tool_context import ToolContext
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.adk.tools.base_tool import BaseTool
from typing import Optional, Dict, Any

Note: you may need to restart the kernel to use updated packages.


In [ ]:
os.environ["GOOGLE_API_KEY"] = "COLOQUE SUA CHAVE AQUI"
os.environ["GROQ_API_KEY"] = "COLOQUE SUA CHAVE AQUI"
os.environ["OLLAMA_API_KEY"] = "COLOQUE SUA CHAVE AQUI"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

In [12]:
session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name="analisador_de_corrida",
    user_id="user_1",
    session_id="sessao_01", 
    state={}
)

In [13]:
CORRIDA_DATABASE =  [
  {
    "id": 1,
    "nome": "Lucas",
    "distancia_km": 10,
    "record_min": 45,
    "meta_min": 39,
    "ultimo_treino_min": 46,
  },
  {
    "id": 2,
    "nome": "Ana",
    "distancia_km": 5,
    "record_min": 24,
    "meta_min": 22,
    "ultimo_treino_min": 25,
  },
  {
    "id": 3,
    "nome": "Carlos",
    "distancia_km": 10,
    "record_min": 50,
    "meta_min": 45,
    "ultimo_treino_min": 52,
  },
  {
    "id": 4,
    "nome": "Marina",
    "distancia_km": 21,
    "record_min": 110,
    "meta_min": 105,
    "ultimo_treino_min": 115,
  },
  {
    "id": 5,
    "nome": "João",
    "distancia_km": 10,
    "record_min": 42,
    "meta_min": 40,
    "ultimo_treino_min": 43,
  }
]

from google.adk.tools import ToolContext

def buscar_usuario(tool_context: ToolContext) -> dict:
    user_id = tool_context.state["user_id"]
    return next(u for u in CORRIDA_DATABASE if u["id"] == user_id)

def calcular_pace(tool_context: ToolContext) -> float:
    usuario = next(u for u in CORRIDA_DATABASE if u["id"] == tool_context.state["user_id"])
    
    pace = usuario["ultimo_treino_min"] / usuario["distancia_km"]  
    tool_context.state["pace"] = pace
    
    return pace

def avaliar_desempenho(tool_context: ToolContext) -> str:
    usuario = next(u for u in CORRIDA_DATABASE if u["id"] == tool_context.state["user_id"])
    
    tempo = usuario["ultimo_treino_min"]
    record = usuario["record_min"]
    meta = usuario["meta_min"]
    
    if tempo < record:
        resultado = "Parabens! Bateu seu RP!"
    elif tempo <= meta:
        resultado = "Foi bom, ficou dentro do planejado."
    else:
        resultado = "Ficou abaixo, vamos trabalhar para melhorar."
    
    tool_context.state["avaliacao"] = resultado
    
    return resultado
    

def gerar_recomendacao(tool_context: ToolContext) -> str:
    avaliacao = tool_context.state["avaliacao"]
    
    if avaliacao == "Ficou abaixo, vamos trabalhar para melhorar":
        return "Precisa de mais velocidade e resistencia, vamos incluir treinos intervalados no seu treinamento."
    else:
        return "Vamos continuar com o plano atual."
    

In [14]:
agente_calculadora = Agent(
    name="agente_calculadora",
    model="gemini-2.5-flash",
    description="Calcula o pace do usuário de acordo com o tempo que foi feito na distância.",
    instruction="Você é um agente responsável por cálculos de corrida. "
                "Quando for necessário calcular o pace de um usuário, utilize a ferramenta 'calcular_pace'. "
                "Antes disso, utilize a ferramenta 'buscar_usuario' para obter os dados necessários. "
                "Utilize apenas as ferramentas disponíveis para realizar os cálculos. "
                "Não explique o resultado, não dê sugestões e não realize análises. "
                "Retorne apenas o resultado do cálculo.",
    tools=[calcular_pace, buscar_usuario],
)

agente_treinador = Agent(
    name="agente_treinador",
    model="gemini-2.5-flash",
    description="Avalia o desempenho do usuário na corrida e fornece recomendações de treino.",
    instruction="Você é um agente treinador de corrida responsável por analisar o desempenho do usuário e sugerir melhorias. "
                "Quando for necessário avaliar o desempenho do usuário, utilize a função 'avaliar_desempenho'. "
                "Quando for necessário gerar recomendações de treino, utilize a função 'gerar_recomendacao'. "
                "Utilize apenas os dados disponíveis no estado da sessão ou retornados pelas ferramentas. "
                "Não invente informações. "
                "Faça apenas essas tarefas de análise e recomendação."
                ,
    tools=[gerar_recomendacao, avaliar_desempenho, buscar_usuario],
)

agente_orquestrador = Agent(
    name="agente_orquestrador",
    model="gemini-2.5-flash",
    description="Responsável por orquestrar o fluxo entre os agentes de corrida, decidindo qual agente deve ser utilizado em cada etapa.",
    instruction="Você é um agente orquestrador responsável por direcionar a requisição do usuário para o agente correto. "
                "Quando o usuário fornecer dados de corrida ou solicitar cálculo de desempenho, utilize o agente 'agente_calculador'. "
                "Quando for necessário analisar o desempenho ou interpretar resultados, utilize o agente 'agente_treinador'. "
                "Sempre utilize os agentes apropriados para cada tarefa, não realizando cálculos ou análises diretamente. "
                "Utilize apenas os agentes disponíveis para responder à requisição. "
                "Faça apenas essa tarefa de orquestração.",
    sub_agents=[agente_calculadora, agente_treinador],
)

runner = Runner(
    agent=agente_orquestrador,
    app_name="analisador_de_corrida",
    session_service=session_service
)


In [15]:
async def call_agente_orquestrador(query: str, user_id: str, session_id: str):
    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    resposta_final = "Nenhuma resposta para apresentar."

    for event in runner.run(
        user_id=user_id,
        session_id=session_id,
        new_message=content
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                resposta_final = event.content.parts[0].text
            break

    return resposta_final

In [16]:
query = "olá!!"
print(await call_agente_orquestrador(query, "user_1", "sessao_01"))

query = "Analise meu treino"
print(await call_agente_orquestrador(query, "user_1", "sessao_01"))

query = "Qual foi meu pace no último treino?"
print(await call_agente_orquestrador(query, "user_1", "sessao_01"))

query = "Meu desempenho foi bom?"
print(await call_agente_orquestrador(query, "user_1", "sessao_01"))

query = "O que posso melhorar no meu treino?"
print(await call_agente_orquestrador(query, "user_1", "sessao_01"))

query = "Analise meu treino e me dê recomendações"
print(await call_agente_orquestrador(query, "user_1", "sessao_01"))

query = "Calcule meu pace"
print(await call_agente_orquestrador(query, "user_1", "sessao_01"))

query = "Corri recentemente, pode avaliar meu desempenho?"
print(await call_agente_orquestrador(query, "user_1", "sessao_01"))

query = "Como foi meu último treino e o que posso fazer para melhorar?"
print(await call_agente_orquestrador(query, "user_1", "sessao_01"))

session2 = await session_service.create_session(
    app_name="analisador_de_corrida",
    user_id="user_2",
    session_id="sessao_02"
)

query = "Analise meu treino"
print(await call_agente_orquestrador(query, "user_2", "sessao_02"))

Olá! Como posso ajudar?


Exception in thread Thread-367 (_asyncio_thread_main):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lucashenriquesiqueirachaves/FASTCAMP_CEIA/venv/lib/python3.13/site-packages/google/adk/runners.py", line 436, in _asyncio_thread_main
    asyncio.run(_invoke_run_async())
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/base_events.py", line 719, in run_until_complete
    return future.result()
           ~~~~~~~~~~~~~^^
  File "/ho

Nenhuma resposta para apresentar.


Exception in thread Thread-369 (_asyncio_thread_main):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lucashenriquesiqueirachaves/FASTCAMP_CEIA/venv/lib/python3.13/site-packages/google/adk/runners.py", line 436, in _asyncio_thread_main
    asyncio.run(_invoke_run_async())
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/base_events.py", line 719, in run_until_complete
    return future.result()
           ~~~~~~~~~~~~~^^
  File "/ho

Nenhuma resposta para apresentar.


...

Nenhuma resposta para apresentar.


Exception in thread Thread-381 (_asyncio_thread_main):
Traceback (most recent call last):
  File "/home/lucashenriquesiqueirachaves/FASTCAMP_CEIA/venv/lib/python3.13/site-packages/google/adk/models/google_llm.py", line 245, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "/home/lucashenriquesiqueirachaves/FASTCAMP_CEIA/venv/lib/python3.13/site-packages/google/genai/models.py", line 7563, in generate_content
    return await self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        model=model, contents=contents, config=parsed_config
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/lucashenriquesiqueirachaves/FASTCAMP_CEIA/venv/lib/python3.13/site-packages/google/genai/models.py", line 6330, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^

Nenhuma resposta para apresentar.


Exception in thread Thread-383 (_asyncio_thread_main):
Traceback (most recent call last):
  File "/home/lucashenriquesiqueirachaves/FASTCAMP_CEIA/venv/lib/python3.13/site-packages/google/adk/models/google_llm.py", line 245, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "/home/lucashenriquesiqueirachaves/FASTCAMP_CEIA/venv/lib/python3.13/site-packages/google/genai/models.py", line 7563, in generate_content
    return await self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        model=model, contents=contents, config=parsed_config
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/lucashenriquesiqueirachaves/FASTCAMP_CEIA/venv/lib/python3.13/site-packages/google/genai/models.py", line 6330, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^

Nenhuma resposta para apresentar.
